# V5 Adaptive Candidate Ranker Training

> Self-contained Colab notebook. No GitHub clone is required.
>
> Paths assume Google Drive layout used throughout this project.

## Goal

Train V5 as a true second-stage ranker on candidates retrieved by V4.

Controlled comparison:

```text
V4 = same candidate set ordered by V4 retrieval score
V5 = same candidate set reranked by V5 rank score
```

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import importlib.util, subprocess, sys
required = ['pyarrow', 'torch', 'pandas', 'numpy']
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('packages ready')

In [ ]:
from __future__ import annotations
from dataclasses import asdict, dataclass
from pathlib import Path
import json, math, random, time
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import torch
from torch import nn
from torch.nn import functional as F

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

In [ ]:
@dataclass
class V5Config:
    adaptive_root: str = '/content/drive/MyDrive/recsys/data/gold/adaptive/v1'
    base_gold_root: str = '/content/drive/MyDrive/recsys/data/gold/two_tower/v1_colab'
    v4_dir: str = '/content/recsys_work/artifacts/adaptive/v4_sequence_retrieval'
    output_dir: str = '/content/recsys_work/artifacts/adaptive/v5_adaptive_ranker'
    seed: int = 43
    smoke_mode: bool = False
    max_train_queries: int | None = 80_000
    max_validation_queries: int | None = 20_000
    max_item_pool_examples: int | None = None
    retrieval_candidates_per_query: int = 1000
    train_candidates_per_query: int = 128
    batch_query_size: int = 128
    epochs: int = 8
    learning_rate: float = 8e-4
    weight_decay: float = 1e-5
    hidden_dims: tuple[int, ...] = (1024, 512, 256)
    dropout: float = 0.15
    eval_top_ks: tuple[int, ...] = (10, 50, 100)
    save_large_cache_to_drive: bool = False
    local_cache_dir: str = '/content/recsys_work/adaptive_cache/v5'

CONFIG = V5Config()
if CONFIG.smoke_mode:
    CONFIG.max_train_queries = 2_000
    CONFIG.max_validation_queries = 500
    CONFIG.max_item_pool_examples = 100_000
    CONFIG.retrieval_candidates_per_query = 300
    CONFIG.train_candidates_per_query = 64
    CONFIG.epochs = 1
OUTPUT_DIR = Path(CONFIG.output_dir); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_CACHE_DIR = Path(CONFIG.local_cache_dir); LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
ADAPTIVE_ROOT = Path(CONFIG.adaptive_root); BASE_GOLD_ROOT = Path(CONFIG.base_gold_root); V4_DIR = Path(CONFIG.v4_dir)
print(json.dumps(asdict(CONFIG), indent=2, default=str))

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed(CONFIG.seed)

def parquet_files(path):
    path = Path(path); files = sorted(p for p in path.rglob('*.parquet') if p.is_file() and not p.name.startswith('.'))
    if not files: raise FileNotFoundError(path)
    return files

def read_parquet(path, columns=None, max_rows=None, filter_expr=None):
    dataset = ds.dataset([str(p) for p in parquet_files(path)], format='parquet')
    if max_rows is None: return dataset.to_table(columns=columns, filter=filter_expr).to_pandas()
    return dataset.head(max_rows, columns=columns, filter=filter_expr).to_pandas()

def save_json(path, payload):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w') as f: json.dump(payload, f, indent=2, sort_keys=True, default=str)

USER_CATEGORICAL=('user_active_degree',); ITEM_CATEGORICAL=('video_type','upload_type'); USER_ID_FEATURES=('user_id',); ITEM_ID_FEATURES=('video_id',)
USER_NUMERIC=('is_lowactive_period','is_live_streamer','is_video_author','follow_user_num','fans_user_num','friend_user_num','register_days',*[f'onehot_feat{i}' for i in range(18)],'user_hist_events','user_hist_long_view_rate','user_hist_like_rate','user_hist_comment_rate','user_hist_forward_rate','user_hist_follow_rate','user_hist_hate_rate','user_hist_avg_watch_ratio','user_hist_avg_play_time_sec','session_event_index','session_elapsed_sec','session_prior_long_view_rate','session_prior_like_rate','session_prior_hate_rate','session_prior_avg_watch_ratio','session_vs_user_long_view_delta','session_vs_user_watch_ratio_delta','event_hour','event_dayofweek','tab','is_rand')
ITEM_NUMERIC=('video_duration_sec','aspect_ratio','visible_status','music_type','upload_age_days_at_event','item_hist_events','item_hist_long_view_rate','item_hist_like_rate','item_hist_hate_rate','item_hist_avg_watch_ratio')
SEQUENCE_COLUMNS=('recent_video_ids','recent_target_classes','recent_engagement_strengths','recent_long_views','recent_likes','recent_hates','recent_clicks','recent_time_ms','recent_event_count','seconds_since_last_event')
TARGET_COLUMNS=('target_class','is_positive','is_observed_negative','is_ambiguous','long_view','is_like','is_follow','is_hate','engagement_strength')
BASE_COLUMNS=('example_id','user_id','video_id','as_of_time','current_time_ms')
INPUT_COLUMNS=list(dict.fromkeys([*BASE_COLUMNS,*TARGET_COLUMNS,*USER_ID_FEATURES,*USER_CATEGORICAL,*USER_NUMERIC,*ITEM_ID_FEATURES,*ITEM_CATEGORICAL,*ITEM_NUMERIC,*SEQUENCE_COLUMNS]))

In [ ]:
def as_sequence(value):
    if value is None:
        return []
    if isinstance(value, float) and pd.isna(value):
        return []
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (list, tuple)):
        return list(value)
    return []


def as_of_time_ms(frame: pd.DataFrame) -> np.ndarray:
    return (pd.to_datetime(frame['as_of_time']).astype('int64') // 1_000_000).to_numpy(dtype='int64')


def sanitize_point_in_time_sequences(frame: pd.DataFrame) -> pd.DataFrame:
    """Drop any sequence events that are not strictly before the target as_of_time.

    This is intentionally conservative and protects training/evaluation even if the
    adaptive data builder attached a same-second history row ambiguously.
    """
    frame = frame.copy()
    boundaries = as_of_time_ms(frame)
    sequence_cols = [
        'recent_video_ids', 'recent_target_classes', 'recent_engagement_strengths',
        'recent_long_views', 'recent_likes', 'recent_hates', 'recent_clicks', 'recent_time_ms',
    ]
    cleaned = {col: [] for col in sequence_cols if col in frame.columns}
    cleaned_counts = []
    dropped_rows = 0
    dropped_events = 0
    for (_, row), boundary in zip(frame.iterrows(), boundaries):
        times = as_sequence(row.get('recent_time_ms'))
        keep = [i for i, t in enumerate(times) if t is not None and not pd.isna(t) and float(t) < float(boundary)]
        if len(keep) != len(times):
            dropped_rows += 1
            dropped_events += len(times) - len(keep)
        for col in cleaned:
            values = as_sequence(row.get(col))
            if len(values) == len(times):
                cleaned[col].append([values[i] for i in keep])
            else:
                cleaned[col].append(values[-len(keep):] if keep else [])
        cleaned_counts.append(len(keep))
    for col, values in cleaned.items():
        frame[col] = values
    frame['recent_event_count'] = cleaned_counts
    frame['current_time_ms'] = boundaries
    print(f'point-in-time sanitize: rows_with_drops={dropped_rows:,} dropped_events={dropped_events:,}')
    return frame

In [ ]:
class Vocabulary:
    def __init__(self, name, values, index=None):
        self.name=name
        if index is None:
            clean=pd.Series(list(values)).dropna().drop_duplicates().tolist(); clean=sorted(clean)
            self.index={v:i+1 for i,v in enumerate(clean)}
        else: self.index=dict(index)
        self.size=max(self.index.values(), default=0)+1
    def encode_many(self, values): return np.asarray([self.index.get(v,0) for v in values], dtype=np.int64)

def load_parquet_vocabulary(path, feature):
    df=read_parquet(path); return Vocabulary(feature, [], dict(zip(df[feature].tolist(), df[f'{feature}_idx'].astype(int).tolist())))

class NumericalPreprocessor:
    def __init__(self, path): self.stats=json.loads(Path(path).read_text()).get('features', json.loads(Path(path).read_text()))
    def transform(self, frame, features):
        outs=[]
        for feat in features:
            vals=pd.to_numeric(frame[feat], errors='coerce').astype('float32') if feat in frame.columns else pd.Series(np.zeros(len(frame)), dtype='float32')
            st=self.stats.get(feat)
            if st:
                vals=vals.fillna(float(st.get('mean',0) or 0)); lo=st.get('p01',st.get('min')); hi=st.get('p99',st.get('max'))
                if lo is not None and hi is not None: vals=vals.clip(float(lo),float(hi))
                std=float(st.get('stddev',0) or 0); mean=float(st.get('mean',0) or 0); vals=(vals-mean)/std if std>1e-6 else vals*0
            outs.append(np.nan_to_num(vals.to_numpy('float32'), nan=0, posinf=0, neginf=0))
        return np.stack(outs, axis=1).astype('float32') if outs else np.zeros((len(frame),0),'float32')

def build_vocabs(train_frame, item_pool):
    return {'user_active_degree':load_parquet_vocabulary(BASE_GOLD_ROOT/'vocabularies/user_active_degree','user_active_degree'),'video_type':load_parquet_vocabulary(BASE_GOLD_ROOT/'vocabularies/video_type','video_type'),'upload_type':load_parquet_vocabulary(BASE_GOLD_ROOT/'vocabularies/upload_type','upload_type'),'user_id':Vocabulary('user_id',train_frame['user_id'].tolist()),'video_id':Vocabulary('video_id',item_pool['video_id'].tolist()),'event_type':Vocabulary('event_type',['AMBIGUOUS_WEAK','OBSERVED_NEGATIVE','STRONG_POSITIVE'])}

In [ ]:
def pad_num(v,max_len,dtype='float32'):
    v=[] if not isinstance(v,(list,tuple,np.ndarray)) else list(v)[-max_len:]
    out=np.zeros(max_len,dtype=dtype)
    if v:
        arr=np.asarray(v,dtype=dtype)
        arr=np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
        out[-len(arr):]=arr
    return out

def pad_enc(v,vocab,max_len):
    v=[] if not isinstance(v,(list,tuple,np.ndarray)) else list(v)[-max_len:]; out=np.zeros(max_len,dtype=np.int64)
    if v: out[-len(v):]=vocab.encode_many(v)[-max_len:]
    return out

def encode_frame(frame, vocabs, numeric, max_len):
    b={}
    for f in USER_ID_FEATURES+USER_CATEGORICAL+ITEM_ID_FEATURES+ITEM_CATEGORICAL: b[f]=torch.as_tensor(vocabs[f].encode_many(frame[f].tolist()), dtype=torch.long)
    b['user_numeric']=torch.as_tensor(numeric.transform(frame, USER_NUMERIC), dtype=torch.float32); b['item_numeric']=torch.as_tensor(numeric.transform(frame, ITEM_NUMERIC), dtype=torch.float32)
    b['seq_video_id']=torch.as_tensor(np.stack([pad_enc(v,vocabs['video_id'],max_len) for v in frame['recent_video_ids']]), dtype=torch.long)
    b['seq_event_type']=torch.as_tensor(np.stack([pad_enc(v,vocabs['event_type'],max_len) for v in frame['recent_target_classes']]), dtype=torch.long)
    sig=np.stack([np.stack([pad_num(v,max_len) for v in frame[c]]) for c in ['recent_engagement_strengths','recent_long_views','recent_likes','recent_hates','recent_clicks']], axis=-1)
    b['seq_signals']=torch.as_tensor(sig, dtype=torch.float32)
    cur=pd.to_numeric(frame['current_time_ms'],errors='coerce').fillna(0).to_numpy('float64'); rec=np.stack([pad_num(v,max_len,'float64') for v in frame['recent_time_ms']])
    age=np.maximum((cur[:,None]-rec)/1000,0); age[rec<=0]=0; b['seq_time_bucket']=torch.as_tensor(np.clip(np.floor(np.log1p(age)/np.log(2)),0,31).astype('int64'))
    b['seq_padding_mask']=b['seq_video_id'].eq(0)
    return b

def move(batch,device): return {k:v.to(device) if torch.is_tensor(v) else v for k,v in batch.items()}

def minimal_user_stub(source_row, n):
    row=source_row
    data={c: row.get(c,0) for c in USER_NUMERIC}
    data.update({'user_id':row['user_id'],'user_active_degree':row.get('user_active_degree'),'recent_video_ids':[row.get('recent_video_ids',[]) for _ in range(n)],'recent_target_classes':[row.get('recent_target_classes',[]) for _ in range(n)],'recent_engagement_strengths':[row.get('recent_engagement_strengths',[]) for _ in range(n)],'recent_long_views':[row.get('recent_long_views',[]) for _ in range(n)],'recent_likes':[row.get('recent_likes',[]) for _ in range(n)],'recent_hates':[row.get('recent_hates',[]) for _ in range(n)],'recent_clicks':[row.get('recent_clicks',[]) for _ in range(n)],'recent_time_ms':[row.get('recent_time_ms',[]) for _ in range(n)],'current_time_ms':row.get('current_time_ms',0)})
    return data

In [ ]:
# V4 model definition must match notebook 12.
def mlp(input_dim, hidden_dims, output_dim, dropout):
    layers=[]; prev=input_dim
    for h in hidden_dims: layers += [nn.Linear(prev,h), nn.ReLU(), nn.Dropout(dropout)]; prev=h
    layers.append(nn.Linear(prev,output_dim)); return nn.Sequential(*layers)

class V4TransformerRetrieval(nn.Module):
    def __init__(self, vocab_sizes, cfg):
        super().__init__(); self.cfg=cfg; rd=cfg['retrieval_dim']; td=cfg['transformer_dim']
        self.user_id_emb=nn.Embedding(vocab_sizes['user_id'],8,padding_idx=0); self.user_active_emb=nn.Embedding(vocab_sizes['user_active_degree'],4,padding_idx=0)
        self.video_id_emb=nn.Embedding(vocab_sizes['video_id'],16,padding_idx=0); self.video_type_emb=nn.Embedding(vocab_sizes['video_type'],3,padding_idx=0); self.upload_type_emb=nn.Embedding(vocab_sizes['upload_type'],8,padding_idx=0)
        self.seq_video_emb=nn.Embedding(vocab_sizes['video_id'],td,padding_idx=0); self.seq_event_emb=nn.Embedding(vocab_sizes['event_type'],32,padding_idx=0); self.seq_event_proj=nn.Linear(32,td); self.seq_signal_proj=nn.Linear(5,td); self.seq_time_emb=nn.Embedding(32,td)
        layer=nn.TransformerEncoderLayer(td,cfg['transformer_heads'],td*4,cfg['transformer_dropout'],batch_first=True,activation='gelu'); self.transformer=nn.TransformerEncoder(layer,cfg['transformer_layers'])
        self.short_proj=nn.Sequential(nn.LayerNorm(td),nn.Linear(td,rd),nn.ReLU()); self.long_user=mlp(8+4+len(USER_NUMERIC),tuple(cfg['user_hidden_dims']),rd,cfg['dropout']); self.item_tower=mlp(16+3+8+len(ITEM_NUMERIC),tuple(cfg['item_hidden_dims']),rd,cfg['dropout']); self.gate=nn.Sequential(nn.Linear(rd*2,rd),nn.Sigmoid())
    def encode_sequence(self,b):
        tok=self.seq_video_emb(b['seq_video_id'])+self.seq_event_proj(self.seq_event_emb(b['seq_event_type']))+self.seq_signal_proj(b['seq_signals'])+self.seq_time_emb(b['seq_time_bucket'].clamp(0,31)); mask=b['seq_padding_mask']; enc=self.transformer(tok,src_key_padding_mask=mask); valid=(~mask).float().unsqueeze(-1); return self.short_proj((enc*valid).sum(1)/valid.sum(1).clamp_min(1))
    def encode_user(self,b):
        long=self.long_user(torch.cat([self.user_id_emb(b['user_id']),self.user_active_emb(b['user_active_degree']),b['user_numeric']],1)); short=self.encode_sequence(b); g=self.gate(torch.cat([long,short],1)); return F.normalize(g*short+(1-g)*long,dim=1)
    def encode_item(self,b):
        return F.normalize(self.item_tower(torch.cat([self.video_id_emb(b['video_id']),self.video_type_emb(b['video_type']),self.upload_type_emb(b['upload_type']),b['item_numeric']],1)),dim=1)

class V5Ranker(nn.Module):
    def __init__(self, retrieval_dim, hidden_dims, dropout):
        super().__init__(); inp=retrieval_dim*4 + 1; self.net=mlp(inp, hidden_dims, 1, dropout)
    def forward(self, user_emb, item_emb, v4_score):
        user_emb=torch.nan_to_num(user_emb, nan=0.0, posinf=0.0, neginf=0.0)
        item_emb=torch.nan_to_num(item_emb, nan=0.0, posinf=0.0, neginf=0.0)
        v4_score=torch.nan_to_num(v4_score, nan=0.0, posinf=1.0, neginf=-1.0).clamp(-1.0,1.0)
        x=torch.cat([user_emb,item_emb,user_emb*item_emb,torch.abs(user_emb-item_emb),v4_score[:,None]],1)
        x=torch.nan_to_num(x, nan=0.0, posinf=1.0, neginf=-1.0)
        return self.net(x).squeeze(1)

In [ ]:
cols=INPUT_COLUMNS
train_queries=read_parquet(ADAPTIVE_ROOT/'sequence_examples/train', cols, CONFIG.max_train_queries, ds.field('target_class')=='STRONG_POSITIVE').reset_index(drop=True)
validation_queries=read_parquet(ADAPTIVE_ROOT/'sequence_examples/validation', cols, CONFIG.max_validation_queries, ds.field('target_class')=='STRONG_POSITIVE').reset_index(drop=True)
train_queries=sanitize_point_in_time_sequences(train_queries)
validation_queries=sanitize_point_in_time_sequences(validation_queries)

ckpt_path = V4_DIR / 'best_model.pt'
assert ckpt_path.exists(), f'Missing V4 checkpoint: {ckpt_path}. Run the V4 section first.'
ckpt=torch.load(ckpt_path, map_location=DEVICE)
v4_cfg=ckpt['config']

# Rebuild V4 vocabularies from the same train-side pools used by V4 training.
# This avoids checkpoint size mismatches such as user_id_emb [995, 8] vs [52, 8].
v4_train_for_vocab=read_parquet(
    ADAPTIVE_ROOT/'sequence_examples/train',
    cols,
    v4_cfg.get('max_train_examples'),
    ds.field('target_class')=='STRONG_POSITIVE',
).reset_index(drop=True)
v4_train_for_vocab=sanitize_point_in_time_sequences(v4_train_for_vocab)
item_pool=read_parquet(
    ADAPTIVE_ROOT/'sequence_examples/train',
    list(dict.fromkeys([*ITEM_ID_FEATURES,*ITEM_CATEGORICAL,*ITEM_NUMERIC])),
    v4_cfg.get('max_item_pool_examples'),
).drop_duplicates('video_id', keep='last').reset_index(drop=True)

print('queries/items:', train_queries.shape, validation_queries.shape, item_pool.shape)
numeric=NumericalPreprocessor(BASE_GOLD_ROOT/'transforms/numeric_stats.json')
vocabs=build_vocabs(v4_train_for_vocab, item_pool)
vocab_sizes=ckpt.get('vocab_sizes', {k:v.size for k,v in vocabs.items()})
print('V4 checkpoint vocab sizes:', vocab_sizes)

v4=V4TransformerRetrieval(vocab_sizes, v4_cfg).to(DEVICE)
v4.load_state_dict(ckpt['model_state_dict'])
v4.eval()
for p in v4.parameters():
    p.requires_grad_(False)
ranker=V5Ranker(v4_cfg['retrieval_dim'], CONFIG.hidden_dims, CONFIG.dropout).to(DEVICE)
opt=torch.optim.AdamW(ranker.parameters(), lr=CONFIG.learning_rate, weight_decay=CONFIG.weight_decay)
loss_fn=nn.BCEWithLogitsLoss()

In [ ]:
def encode_items(frame, batch_size=8192):
    embs=[]; ids=[]
    with torch.no_grad():
        for s in range(0,len(frame),batch_size):
            part=frame.iloc[s:s+batch_size].reset_index(drop=True); dummy=part.assign(**minimal_user_stub(train_queries.iloc[0].to_dict(), len(part)))
            emb=v4.encode_item(move(encode_frame(dummy,vocabs,numeric,v4_cfg['sequence_max_len']),DEVICE)).cpu().numpy(); embs.append(emb); ids.append(part['video_id'].to_numpy('int64'))
    return np.concatenate(ids), np.vstack(embs).astype('float32')

item_ids,item_emb=np.array([]),np.empty((0,v4_cfg['retrieval_dim']),'float32')
item_ids,item_emb=encode_items(item_pool)
item_index=pd.DataFrame({'video_id':item_ids, 'item_pos':np.arange(len(item_ids))})
print('item embedding matrix:', item_emb.shape)


def retrieve_candidates(query_batch: pd.DataFrame, top_k: int):
    with torch.no_grad():
        enc=move(encode_frame(query_batch,vocabs,numeric,v4_cfg['sequence_max_len']),DEVICE); user_emb=v4.encode_user(enc).cpu().numpy()
    scores=user_emb @ item_emb.T; k=min(top_k, scores.shape[1])
    idx=np.argpartition(-scores, kth=k-1, axis=1)[:,:k]; vals=np.take_along_axis(scores,idx,axis=1); order=np.argsort(-vals,axis=1); idx=np.take_along_axis(idx,order,axis=1); vals=np.take_along_axis(vals,order,axis=1)
    return user_emb.astype('float32'), idx, vals.astype('float32')


def candidate_training_batch(query_batch: pd.DataFrame):
    user_emb, idx, vals = retrieve_candidates(query_batch, CONFIG.retrieval_candidates_per_query)
    rows=[]; labels=[]; uemb=[]; iemb=[]; scores=[]
    for qi,row in query_batch.reset_index(drop=True).iterrows():
        cand_idx=idx[qi,:CONFIG.train_candidates_per_query].copy(); cand_scores=vals[qi,:CONFIG.train_candidates_per_query].copy(); true_vid=int(row['video_id'])
        true_positions=np.where(item_ids==true_vid)[0]
        if len(true_positions) and true_vid not in item_ids[cand_idx]:
            cand_idx[-1]=true_positions[0]; cand_scores[-1]=float(user_emb[qi] @ item_emb[true_positions[0]])
        for ci,sc in zip(cand_idx,cand_scores):
            uemb.append(user_emb[qi]); iemb.append(item_emb[ci]); scores.append(sc); labels.append(1.0 if int(item_ids[ci])==true_vid else 0.0)
    uarr=np.nan_to_num(np.vstack(uemb).astype('float32'), nan=0.0, posinf=0.0, neginf=0.0)
    iarr=np.nan_to_num(np.vstack(iemb).astype('float32'), nan=0.0, posinf=0.0, neginf=0.0)
    sarr=np.nan_to_num(np.asarray(scores,dtype='float32'), nan=0.0, posinf=1.0, neginf=-1.0)
    sarr=np.clip(sarr, -1.0, 1.0)
    yarr=np.asarray(labels,dtype='float32')
    return torch.as_tensor(uarr,device=DEVICE), torch.as_tensor(iarr,device=DEVICE), torch.as_tensor(sarr,device=DEVICE,dtype=torch.float32), torch.as_tensor(yarr,device=DEVICE,dtype=torch.float32)

In [ ]:
history=[]
best_loss=float('inf')
best_epoch=None
for epoch in range(1, CONFIG.epochs+1):
    ranker.train()
    order=np.random.permutation(len(train_queries))
    total=0
    loss_sum=0.0
    for start in range(0,len(order),CONFIG.batch_query_size):
        qb=train_queries.iloc[order[start:start+CONFIG.batch_query_size]].reset_index(drop=True)
        ue,ie,sc,y=candidate_training_batch(qb)
        logits=ranker(ue,ie,sc)
        if not torch.isfinite(logits).all():
            raise RuntimeError('Non-finite V5 logits detected. Check candidate_training_batch sanitization.')
        loss=loss_fn(logits,y)
        if not torch.isfinite(loss):
            print('bad batch diagnostics:', {
                'ue_finite': bool(torch.isfinite(ue).all().item()),
                'ie_finite': bool(torch.isfinite(ie).all().item()),
                'score_finite': bool(torch.isfinite(sc).all().item()),
                'label_mean': float(y.mean().item()),
                'score_min': float(sc.min().item()),
                'score_max': float(sc.max().item()),
            })
            raise RuntimeError('Non-finite V5 loss detected')
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(ranker.parameters(),5)
        opt.step()
        loss_sum += loss.item()*len(y)
        total += len(y)
        if (start//CONFIG.batch_query_size+1)%20==0:
            print('epoch',epoch,'batch',start//CONFIG.batch_query_size+1,'loss',loss_sum/max(total,1),flush=True)
    train_loss=loss_sum/max(total,1)
    row={'epoch':epoch, 'train_loss':train_loss}
    history.append(row)
    print(row, flush=True)
    if train_loss < best_loss:
        best_loss=train_loss
        best_epoch=epoch
        torch.save({
            'model_state_dict':ranker.state_dict(),
            'config':asdict(CONFIG),
            'v4_config':v4_cfg,
            'best_epoch':best_epoch,
            'best_train_loss':best_loss,
        }, OUTPUT_DIR/'best_model.pt')
        print('saved V5 best_model.pt:', OUTPUT_DIR/'best_model.pt', flush=True)

pd.DataFrame(history).to_csv(OUTPUT_DIR/'training_history.csv',index=False)
assert (OUTPUT_DIR/'best_model.pt').exists(), f'V5 checkpoint was not saved: {OUTPUT_DIR/"best_model.pt"}'
print('best_epoch:', best_epoch, 'best_train_loss:', best_loss)

In [ ]:
def binary_diagnostics(labels, scores):
    labels=np.asarray(labels, dtype=np.float64); scores=np.asarray(scores, dtype=np.float64)
    probs=1/(1+np.exp(-np.clip(scores, -30, 30)))
    eps=1e-7
    logloss=float(-(labels*np.log(probs+eps)+(1-labels)*np.log(1-probs+eps)).mean()) if len(labels) else None
    pos=int(labels.sum()); neg=int(len(labels)-pos)
    if pos and neg:
        order=np.argsort(scores)
        ranks=np.empty_like(order, dtype=np.float64); ranks[order]=np.arange(1, len(scores)+1)
        auc=float((ranks[labels==1].sum()-pos*(pos+1)/2)/(pos*neg))
    else:
        auc=None
    if pos:
        desc=np.argsort(-scores); sorted_labels=labels[desc]
        precision=np.cumsum(sorted_labels)/(np.arange(len(sorted_labels))+1)
        ap=float((precision*sorted_labels).sum()/pos)
    else:
        ap=None
    return {'auc':auc, 'logloss':logloss, 'average_precision':ap, 'positives':pos, 'examples':int(len(labels))}


def ranking_metrics_for_queries(query_frame: pd.DataFrame, limit=None):
    ranker.eval(); frame=query_frame.head(limit).reset_index(drop=True) if limit else query_frame.reset_index(drop=True); rows=[]; point_labels=[]; point_scores=[]
    for start in range(0,len(frame),CONFIG.batch_query_size):
        qb=frame.iloc[start:start+CONFIG.batch_query_size].reset_index(drop=True); ue,idx,vals=retrieve_candidates(qb, CONFIG.retrieval_candidates_per_query)
        flat_u=[]; flat_i=[]; flat_s=[]
        for qi in range(len(qb)):
            for ci,sc in zip(idx[qi], vals[qi]): flat_u.append(ue[qi]); flat_i.append(item_emb[ci]); flat_s.append(sc)
        with torch.no_grad():
            v5_scores=ranker(torch.as_tensor(np.vstack(flat_u),device=DEVICE), torch.as_tensor(np.vstack(flat_i),device=DEVICE), torch.as_tensor(flat_s,device=DEVICE,dtype=torch.float32)).cpu().numpy()
        offset=0
        for qi,row in qb.iterrows():
            n=idx.shape[1]; vids=item_ids[idx[qi]]; v4_scores=vals[qi]; rerank=v5_scores[offset:offset+n]; offset+=n; true=int(row['video_id'])
            labels=(vids==true).astype('float32')
            point_labels.extend(labels.tolist()); point_scores.extend(rerank.tolist())
            rows.append({'true_video_id':true,'v4_ranked':vids[np.argsort(-v4_scores)].tolist(),'v5_ranked':vids[np.argsort(-rerank)].tolist()})
    def compute(which):
        out={}; ks=CONFIG.eval_top_ks
        for k in ks:
            hits=[]; nd=[]
            for r in rows:
                arr=np.asarray(r[which][:k]); pos=np.where(arr==r['true_video_id'])[0]; hits.append(len(pos)>0); nd.append(0 if len(pos)==0 else 1/math.log2(int(pos[0])+2))
            out[f'hitrate@{k}']=float(np.mean(hits)); out[f'ndcg@{k}']=float(np.mean(nd))
        return out
    return {'v4_ordering':compute('v4_ranked'), 'v5_reranked':compute('v5_ranked'), 'ranker_pointwise':binary_diagnostics(point_labels, point_scores), 'evaluated_queries':len(rows)}

metrics=ranking_metrics_for_queries(validation_queries)
print(json.dumps(metrics, indent=2))
save_json(OUTPUT_DIR/'metrics.json', {'best_epoch':best_epoch,'ranking_comparison':metrics,'history':history})
save_json(OUTPUT_DIR/'config.json', asdict(CONFIG))
save_json(OUTPUT_DIR/'v4_candidate_training_manifest.json', {'retrieval_candidates_per_query':CONFIG.retrieval_candidates_per_query,'train_candidates_per_query':CONFIG.train_candidates_per_query,'candidate_source':'V4 exact matrix search'})

## Download Local V5 Artifacts

Artifacts are written to Colab local disk. Run this cell after training to download them to your machine.

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive('/content/v5_adaptive_ranker', 'zip', str(OUTPUT_DIR))
print('created', zip_path)
files.download(zip_path)